In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install ipykernel numpy pandas networkx matplotlib scipy openpyxl

In [ ]:
"""
=============================================================================
ANÁLISIS DE REDES COMPLEJAS PARA ECG - JUPYTER NOTEBOOK
=============================================================================
Adaptado para: MacBook Pro Late 2011 / macOS High Sierra
Ruta de datos: /Users/noelozanovazquez/AN2026/Datos
Autores: Noé Lozano y Eunice Hernández
=============================================================================
"""

# Configuración para Jupyter
%matplotlib inline

# Importaciones
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.gridspec import GridSpec
import warnings
from pathlib import Path
from scipy import stats
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import os
import time

# Suprimir warnings
warnings.filterwarnings('ignore')

# Configuración de matplotlib para mejor rendimiento en Mac antiguo
matplotlib.use('Agg')  # Backend más ligero
plt.rcParams['figure.max_open_warning'] = 50
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10

# Intentar importar seaborn (opcional)
try:
    import seaborn as sns
    sns.set_style("whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print("⚠️ Seaborn no instalado. Usando matplotlib básico.")

# Intentar importar tqdm para barras de progreso
try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    try:
        from tqdm import tqdm
        HAS_TQDM = True
    except ImportError:
        HAS_TQDM = False
        print("⚠️ tqdm no instalado. No se mostrarán barras de progreso.")
        def tqdm(x, **kwargs):
            return x

print("✅ Importaciones completadas")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   NetworkX: {nx.__version__}")
print(f"   Matplotlib: {matplotlib.__version__}")

✅ Importaciones completadas
   NumPy: 2.0.2
   Pandas: 2.2.2
   NetworkX: 3.6.1
   Matplotlib: 3.10.0


In [ ]:
"""
=============================================================================
CONFIGURACIÓN DE RUTAS Y PARÁMETROS
=============================================================================
"""

# ======================= MODIFICAR AQUÍ SI ES NECESARIO =======================
BASE_PATH = '/content/drive/MyDrive/8vo_LiFTA/Análisis_Numérico/AN2026/Datos3'
RESULTS_PATH = '/content/drive/MyDrive/8vo_LiFTA/Análisis_Numérico/AN2026/Resultados3'
# =============================================================================

# Parámetros del análisis
WINDOW_SIZE = 650  # Tamaño de ventana para segmentar datos de Sleep
GRAPH_TYPE = 'visibility'  # 'visibility' o 'horizontal'

# Crear directorio de resultados
os.makedirs(RESULTS_PATH, exist_ok=True)

# Verificar que existen las carpetas de datos
carpetas = ['Sleep', 'TaiChi', 'Yoga']
print("📁 Verificando estructura de datos...")
print(f"   Ruta base: {BASE_PATH}")
print()

for carpeta in carpetas:
    ruta = Path(BASE_PATH) / carpeta
    if ruta.exists():
        archivos = list(ruta.glob('*.txt'))
        print(f"   ✅ {carpeta}: {len(archivos)} archivos encontrados")
        for f in sorted(archivos):
            print(f"      - {f.name}")
    else:
        print(f"   ❌ {carpeta}: NO ENCONTRADA")

print()
print(f"📁 Resultados se guardarán en: {RESULTS_PATH}")

📁 Verificando estructura de datos...
   Ruta base: /content/drive/MyDrive/8vo_LiFTA/Análisis_Numérico/AN2026/Datos3

   ✅ Sleep: 8 archivos encontrados
      - sleep1.txt
      - sleep2.txt
      - sleep3.txt
      - sleep4.txt
      - sleep5.txt
      - sleep6.txt
      - sleep7.txt
      - sleep8.txt
   ✅ TaiChi: 4 archivos encontrados
      - taichi1.txt
      - taichi2.txt
      - taichi3.txt
      - taichi4.txt
   ✅ Yoga: 4 archivos encontrados
      - yoga1.txt
      - yoga2.txt
      - yoga3.txt
      - yoga4.txt

📁 Resultados se guardarán en: /content/drive/MyDrive/8vo_LiFTA/Análisis_Numérico/AN2026/Resultados3


In [ ]:
"""
=============================================================================
CLASES Y FUNCIONES PARA MANEJO DE DATOS
=============================================================================
"""

@dataclass
class NetworkMetrics:
    """Contenedor para métricas de red."""
    num_nodes: int
    num_edges: int
    density: float
    avg_degree: float
    avg_clustering: float
    avg_path_length: float
    diameter: int
    assortativity: float
    transitivity: float
    avg_betweenness: float
    avg_closeness: float

    def to_dict(self) -> dict:
        return {
            'Nodos': self.num_nodes,
            'Aristas': self.num_edges,
            'Densidad': self.density,
            'Grado_Promedio': self.avg_degree,
            'Clustering_Promedio': self.avg_clustering,
            'Longitud_Camino_Promedio': self.avg_path_length,
            'Diametro': self.diameter,
            'Asortatividad': self.assortativity,
            'Transitividad': self.transitivity,
            'Betweenness_Promedio': self.avg_betweenness,
            'Closeness_Promedio': self.avg_closeness
        }


def cargar_ecg(filepath: str) -> np.ndarray:
    """Carga datos de ECG desde archivo .txt."""
    try:
        data = np.loadtxt(filepath)
        return data.flatten()
    except Exception as e:
        raise IOError(f"Error al cargar {filepath}: {e}")


def segmentar_señal(signal: np.ndarray, window_size: int = 650) -> List[np.ndarray]:
    """Segmenta una señal en ventanas de tamaño fijo."""
    num_windows = len(signal) // window_size
    segments = []
    for i in range(num_windows):
        start = i * window_size
        end = start + window_size
        segments.append(signal[start:end])
    return segments


def cargar_todos_los_datos(base_path: str, window_size: int = 650) -> Dict:
    """
    Carga todos los datos de los tres grupos.

    Estructura esperada:
    - Sleep/sleep1.txt ... sleep8.txt (20,000 puntos cada uno)
    - TaiChi/taichi1.txt ... taichi4.txt (650 puntos cada uno)
    - Yoga/yoga1.txt ... yoga4.txt (650 puntos cada uno)
    """
    data = {
        'taichi': {},
        'yoga': {},
        'sleep': {}
    }

    print("\n" + "="*60)
    print("CARGANDO DATOS")
    print("="*60)

    # Cargar datos de TaiChi
    taichi_path = Path(base_path) / 'TaiChi'
    if taichi_path.exists():
        for i in range(1, 5):
            file = taichi_path / f'taichi{i}.txt'
            if file.exists():
                ecg = cargar_ecg(str(file))
                data['taichi'][f'taichi{i}'] = ecg
                print(f"✓ Cargado: taichi{i}.txt ({len(ecg)} puntos)")

    # Cargar datos de Yoga
    yoga_path = Path(base_path) / 'Yoga'
    if yoga_path.exists():
        for i in range(1, 5):
            file = yoga_path / f'yoga{i}.txt'
            if file.exists():
                ecg = cargar_ecg(str(file))
                data['yoga'][f'yoga{i}'] = ecg
                print(f"✓ Cargado: yoga{i}.txt ({len(ecg)} puntos)")

    # Cargar y segmentar datos de Sleep (control)
    sleep_path = Path(base_path) / 'Sleep'
    if sleep_path.exists():
        for i in range(1, 9):
            file = sleep_path / f'sleep{i}.txt'
            if file.exists():
                ecg = cargar_ecg(str(file))
                segments = segmentar_señal(ecg, window_size)
                data['sleep'][f'sleep{i}'] = {
                    'original': ecg,
                    'segments': segments,
                    'num_points': len(ecg)
                }
                print(f"✓ Cargado: sleep{i}.txt ({len(ecg)} puntos → {len(segments)} ventanas de {window_size})")

    print("\n" + "-"*60)
    print(f"RESUMEN: TaiChi={len(data['taichi'])}, Yoga={len(data['yoga'])}, Sleep={len(data['sleep'])}")
    print("="*60 + "\n")

    return data


print("✅ Funciones de datos definidas")

✅ Funciones de datos definidas


In [ ]:
"""
=============================================================================
CONSTRUCCIÓN DE GRAFOS DE VISIBILIDAD
=============================================================================
"""

def visibility_graph(series: np.ndarray) -> nx.Graph:
    """
    Construye un grafo de visibilidad natural (NVG).

    Dos nodos i y j están conectados si todos los puntos intermedios k cumplen:
    y_k < y_i + (y_j - y_i) * (k - i) / (j - i)
    """
    n = len(series)
    G = nx.Graph()
    G.add_nodes_from(range(n))

    # Añadir atributos de nodo
    for i in range(n):
        G.nodes[i]['value'] = series[i]
        G.nodes[i]['time'] = i

    # Construir aristas basadas en visibilidad
    for i in range(n - 1):
        for j in range(i + 1, n):
            visible = True
            for k in range(i + 1, j):
                y_line = series[i] + (series[j] - series[i]) * (k - i) / (j - i)
                if series[k] >= y_line:
                    visible = False
                    break
            if visible:
                G.add_edge(i, j)

    return G


def horizontal_visibility_graph(series: np.ndarray) -> nx.Graph:
    """
    Construye un grafo de visibilidad horizontal (HVG).

    Dos nodos i y j están conectados si todos los puntos intermedios k cumplen:
    y_k < min(y_i, y_j)
    """
    n = len(series)
    G = nx.Graph()
    G.add_nodes_from(range(n))

    for i in range(n):
        G.nodes[i]['value'] = series[i]
        G.nodes[i]['time'] = i

    for i in range(n - 1):
        for j in range(i + 1, n):
            min_val = min(series[i], series[j])
            visible = True
            for k in range(i + 1, j):
                if series[k] >= min_val:
                    visible = False
                    break
            if visible:
                G.add_edge(i, j)

    return G


def construir_grafo(series: np.ndarray, graph_type: str = 'visibility') -> nx.Graph:
    """Wrapper para construir el grafo según el tipo especificado."""
    if graph_type == 'horizontal':
        return horizontal_visibility_graph(series)
    return visibility_graph(series)


print("✅ Funciones de construcción de grafos definidas")

✅ Funciones de construcción de grafos definidas


In [ ]:
"""
=============================================================================
CÁLCULO DE MÉTRICAS DE RED
=============================================================================
"""

def calcular_metricas(G: nx.Graph) -> NetworkMetrics:
    """Calcula métricas comprehensivas de una red."""

    # Métricas básicas
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    density = nx.density(G)

    # Grado
    degrees = [d for n, d in G.degree()]
    avg_degree = np.mean(degrees) if degrees else 0

    # Clustering
    avg_clustering = nx.average_clustering(G)
    transitivity = nx.transitivity(G)

    # Caminos (componente conectado más grande)
    if nx.is_connected(G):
        largest_cc = G
    else:
        largest_cc = G.subgraph(max(nx.connected_components(G), key=len)).copy()

    if largest_cc.number_of_nodes() > 1:
        try:
            avg_path_length = nx.average_shortest_path_length(largest_cc)
            diameter = nx.diameter(largest_cc)
        except:
            avg_path_length = 0
            diameter = 0
    else:
        avg_path_length = 0
        diameter = 0

    # Asortatividad
    try:
        assortativity = nx.degree_assortativity_coefficient(G)
        if np.isnan(assortativity):
            assortativity = 0
    except:
        assortativity = 0

    # Centralidades (con manejo de errores para redes pequeñas)
    try:
        betweenness = nx.betweenness_centrality(G)
        avg_betweenness = np.mean(list(betweenness.values()))
    except:
        avg_betweenness = 0

    try:
        closeness = nx.closeness_centrality(G)
        avg_closeness = np.mean(list(closeness.values()))
    except:
        avg_closeness = 0

    return NetworkMetrics(
        num_nodes=num_nodes,
        num_edges=num_edges,
        density=density,
        avg_degree=avg_degree,
        avg_clustering=avg_clustering,
        avg_path_length=avg_path_length,
        diameter=diameter,
        assortativity=assortativity,
        transitivity=transitivity,
        avg_betweenness=avg_betweenness,
        avg_closeness=avg_closeness
    )


def calcular_metricas_extendidas(G: nx.Graph) -> Dict:
    """Calcula métricas adicionales de la red."""
    degrees = [d for n, d in G.degree()]

    metricas = {
        'grado_max': max(degrees) if degrees else 0,
        'grado_min': min(degrees) if degrees else 0,
        'grado_std': np.std(degrees) if degrees else 0,
        'num_componentes': nx.number_connected_components(G),
    }

    # Eficiencias (pueden ser costosas computacionalmente)
    try:
        metricas['eficiencia_global'] = nx.global_efficiency(G)
    except:
        metricas['eficiencia_global'] = 0

    return metricas


print("✅ Funciones de métricas definidas")

✅ Funciones de métricas definidas


In [ ]:
"""
=============================================================================
CARGA DE DATOS
=============================================================================
"""

# Cargar todos los datos
data = cargar_todos_los_datos(BASE_PATH, WINDOW_SIZE)

# Mostrar información detallada
print("\n📊 INFORMACIÓN DETALLADA DE LOS DATOS:")
print("-"*60)

print("\n🧘 MEDITADORES TAI-CHI:")
for name, signal in data['taichi'].items():
    print(f"   {name}: {len(signal)} puntos | Rango: [{signal.min():.2f}, {signal.max():.2f}]")

print("\n🧘 MEDITADORES YOGA:")
for name, signal in data['yoga'].items():
    print(f"   {name}: {len(signal)} puntos | Rango: [{signal.min():.2f}, {signal.max():.2f}]")

print("\n😴 GRUPO SLEEP (CONTROL):")
for name, info in data['sleep'].items():
    print(f"   {name}: {info['num_points']} puntos → {len(info['segments'])} ventanas")


CARGANDO DATOS
✓ Cargado: taichi1.txt (4034 puntos)
✓ Cargado: taichi2.txt (3599 puntos)
✓ Cargado: taichi3.txt (3767 puntos)
✓ Cargado: taichi4.txt (3150 puntos)
✓ Cargado: yoga1.txt (1022 puntos)
✓ Cargado: yoga2.txt (1127 puntos)
✓ Cargado: yoga3.txt (910 puntos)
✓ Cargado: yoga4.txt (933 puntos)
✓ Cargado: sleep1.txt (864 puntos → 1 ventanas de 650)
✓ Cargado: sleep2.txt (906 puntos → 1 ventanas de 650)
✓ Cargado: sleep3.txt (841 puntos → 1 ventanas de 650)
✓ Cargado: sleep4.txt (963 puntos → 1 ventanas de 650)
✓ Cargado: sleep5.txt (661 puntos → 1 ventanas de 650)
✓ Cargado: sleep6.txt (830 puntos → 1 ventanas de 650)
✓ Cargado: sleep7.txt (1266 puntos → 1 ventanas de 650)
✓ Cargado: sleep8.txt (899 puntos → 1 ventanas de 650)

------------------------------------------------------------
RESUMEN: TaiChi=4, Yoga=4, Sleep=8


📊 INFORMACIÓN DETALLADA DE LOS DATOS:
------------------------------------------------------------

🧘 MEDITADORES TAI-CHI:
   taichi1: 4034 puntos | Rango: [6

In [ ]:
"""
=============================================================================
CELDA 6b: CORTAR DATOS DE MEDITADORES A 650 PUNTOS
=============================================================================
"""

PUNTOS_CORTE = 650

print("="*60)
print(f"CORTANDO DATOS DE MEDITADORES A {PUNTOS_CORTE} PUNTOS")
print("="*60)

# Cortar TaiChi
print("\n🟠 TAI-CHI:")
for name in data['taichi'].keys():
    original = len(data['taichi'][name])
    data['taichi'][name] = data['taichi'][name][:PUNTOS_CORTE]
    nuevo = len(data['taichi'][name])
    print(f"   {name}: {original} → {nuevo} puntos")

# Cortar Yoga
print("\n🟢 YOGA:")
for name in data['yoga'].keys():
    original = len(data['yoga'][name])
    data['yoga'][name] = data['yoga'][name][:PUNTOS_CORTE]
    nuevo = len(data['yoga'][name])
    print(f"   {name}: {original} → {nuevo} puntos")

# Verificar Sleep (ya debería estar en ventanas de 650)
print("\n🔵 SLEEP (verificación):")
for name in data['sleep'].keys():
    ventanas = len(data['sleep'][name]['segments'])
    puntos_ventana = len(data['sleep'][name]['segments'][0])
    print(f"   {name}: {ventanas} ventanas de {puntos_ventana} puntos")

print("\n" + "="*60)
print("✅ Todos los datos tienen 650 puntos")
print("="*60)

CORTANDO DATOS DE MEDITADORES A 650 PUNTOS

🟠 TAI-CHI:
   taichi1: 4034 → 650 puntos
   taichi2: 3599 → 650 puntos
   taichi3: 3767 → 650 puntos
   taichi4: 3150 → 650 puntos

🟢 YOGA:
   yoga1: 1022 → 650 puntos
   yoga2: 1127 → 650 puntos
   yoga3: 910 → 650 puntos
   yoga4: 933 → 650 puntos

🔵 SLEEP (verificación):
   sleep1: 1 ventanas de 650 puntos
   sleep2: 1 ventanas de 650 puntos
   sleep3: 1 ventanas de 650 puntos
   sleep4: 1 ventanas de 650 puntos
   sleep5: 1 ventanas de 650 puntos
   sleep6: 1 ventanas de 650 puntos
   sleep7: 1 ventanas de 650 puntos
   sleep8: 1 ventanas de 650 puntos

✅ Todos los datos tienen 650 puntos


In [ ]:
"""
=============================================================================
CELDA 7: VISUALIZACIÓN DE SEÑALES ECG ORIGINALES
=============================================================================
"""

# Colores unificados
COLOR_TAICHI = '#F4A59B'  # Naranja
COLOR_YOGA = '#A1D99B'    # Verde
COLOR_SLEEP = '#92C5DE'   # Azul

fig, axes = plt.subplots(4, 4, figsize=(16, 12))

# TaiChi (naranja)
for i, (name, signal) in enumerate(data['taichi'].items()):
    ax = axes[0, i]
    ax.plot(signal, color=COLOR_TAICHI, linewidth=0.5)
    ax.set_title(f'{name}', fontsize=10)
    ax.set_xlabel('Muestras')
    ax.set_ylabel('Amplitud')
    ax.grid(True, alpha=0.3)

# Yoga (verde)
for i, (name, signal) in enumerate(data['yoga'].items()):
    ax = axes[1, i]
    ax.plot(signal, color=COLOR_YOGA, linewidth=0.5)
    ax.set_title(f'{name}', fontsize=10)
    ax.set_xlabel('Muestras')
    ax.set_ylabel('Amplitud')
    ax.grid(True, alpha=0.3)

# Sleep (azul) - primeros 4
for i, (name, info) in enumerate(list(data['sleep'].items())[:4]):
    ax = axes[2, i]
    ax.plot(info['segments'][0], color=COLOR_SLEEP, linewidth=0.5)
    ax.set_title(f'{name} (ventana 1)', fontsize=10)
    ax.set_xlabel('Muestras')
    ax.set_ylabel('Amplitud')
    ax.grid(True, alpha=0.3)

# Sleep (azul) - últimos 4
for i, (name, info) in enumerate(list(data['sleep'].items())[4:8]):
    ax = axes[3, i]
    ax.plot(info['segments'][0], color=COLOR_SLEEP, linewidth=0.5)
    ax.set_title(f'{name} (ventana 1)', fontsize=10)
    ax.set_xlabel('Muestras')
    ax.set_ylabel('Amplitud')
    ax.grid(True, alpha=0.3)

plt.suptitle('Señales ECG Originales\n🟠 Tai-Chi  🟢 Yoga  🔵 Sleep', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/01_senales_ecg_originales.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/01_senales_ecg_originales.png")

✅ Figura guardada: /content/drive/MyDrive/8vo_LiFTA/Análisis_Numérico/AN2026/Resultados3/01_senales_ecg_originales.png


In [ ]:
"""
=============================================================================
CONSTRUCCIÓN Y VISUALIZACIÓN DE REDES - MEDITADORES
=============================================================================
"""

print("🔄 Construyendo redes de visibilidad para meditadores...")
print("   (Esto puede tomar unos minutos...)\n")

# Almacenar grafos y métricas
grafos_meditadores = {}
metricas_meditadores = {}

# Construir grafos para TaiChi
print("📍 Procesando TaiChi...")
for name, signal in data['taichi'].items():
    start_time = time.time()
    G = construir_grafo(signal, GRAPH_TYPE)
    metrics = calcular_metricas(G)
    grafos_meditadores[name] = G
    metricas_meditadores[name] = metrics
    print(f"   ✓ {name}: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas ({time.time()-start_time:.1f}s)")

# Construir grafos para Yoga
print("\n📍 Procesando Yoga...")
for name, signal in data['yoga'].items():
    start_time = time.time()
    G = construir_grafo(signal, GRAPH_TYPE)
    metrics = calcular_metricas(G)
    grafos_meditadores[name] = G
    metricas_meditadores[name] = metrics
    print(f"   ✓ {name}: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas ({time.time()-start_time:.1f}s)")

print("\n✅ Redes de meditadores construidas")

🔄 Construyendo redes de visibilidad para meditadores...
   (Esto puede tomar unos minutos...)

📍 Procesando TaiChi...


In [ ]:
"""
=============================================================================
CONSTRUIR REDES DE VISIBILIDAD - SLEEP (CONTROL)
=============================================================================
"""

print("🔄 Construyendo redes de visibilidad para grupo Sleep...")
print("   (Esto tomará varios minutos debido a la cantidad de ventanas...)\n")

# Almacenar grafos y métricas de Sleep
grafos_sleep = {}
metricas_sleep = {}

total_ventanas = sum(len(info['segments']) for info in data['sleep'].values())
print(f"   Total de ventanas a procesar: {total_ventanas}")
print("-"*60)

for name, info in data['sleep'].items():
    grafos_sleep[name] = []
    metricas_sleep[name] = []

    print(f"\n🔵 Procesando {name} ({len(info['segments'])} ventanas)...")

    for j, segment in enumerate(info['segments']):
        G = construir_grafo(segment, GRAPH_TYPE)
        metrics = calcular_metricas(G)
        grafos_sleep[name].append(G)
        metricas_sleep[name].append(metrics)

        # Mostrar progreso cada 5 ventanas
        if (j + 1) % 5 == 0 or j == len(info['segments']) - 1:
            print(f"   [{j+1}/{len(info['segments'])}] completadas")

print("\n" + "="*60)
print("✅ Redes de Sleep (control) construidas")
print(f"   Total de grafos generados: {sum(len(v) for v in grafos_sleep.values())}")

In [ ]:
"""
=============================================================================
CELDA 8b: FUNCIÓN PARA GENERAR REDES (CON PARÁMETRO DE COLOR)
=============================================================================
"""

def visualizar_red_completa(grafo, titulo, filename, output_dir, color_nodos):
    """Genera red y guarda como PNG."""

    os.makedirs(output_dir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(12, 12))

    pos = nx.spring_layout(grafo, iterations=15, seed=42)

    nx.draw(grafo, pos, ax=ax,
            node_size=10,
            node_color=color_nodos,
            edge_color='gray',
            alpha=0.3,
            with_labels=False)

    ax.set_title(titulo, fontsize=14, fontweight='bold')

    filepath = os.path.join(output_dir, f'{filename}.png')
    plt.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()

    return filepath


print("✅ Función actualizada con parámetro de color")

In [ ]:
"""
=============================================================================
CELDA 8c: GENERAR REDES - MEDITADORES
=============================================================================
"""

print("="*80)
print("GENERANDO REDES DE MEDITADORES")
print("="*80)

redes_taichi_dir = f'{RESULTS_PATH}/redes_taichi'
redes_yoga_dir = f'{RESULTS_PATH}/redes_yoga'

# TAI-CHI (NARANJA)
print("\n🟠 TAI-CHI (naranja):")
for i, name in enumerate(data['taichi'].keys(), 1):
    G = grafos_meditadores[name]
    visualizar_red_completa(G, f'{name.upper()} (Tai-Chi)', f'red_{name}', redes_taichi_dir, '#F4A59B')
    print(f"   [{i}/4] ✓ {name}")

# YOGA (VERDE)
print("\n🟢 YOGA (verde):")
for i, name in enumerate(data['yoga'].keys(), 1):
    G = grafos_meditadores[name]
    visualizar_red_completa(G, f'{name.upper()} (Yoga)', f'red_{name}', redes_yoga_dir, '#A1D99B')
    print(f"   [{i}/4] ✓ {name}")

print(f"\n📁 Tai-Chi: {redes_taichi_dir}")
print(f"📁 Yoga: {redes_yoga_dir}")
print("\n✅ Meditadores completados")

In [ ]:
"""
=============================================================================
CELDA 8d: GENERAR REDES - SLEEP (AZUL)
=============================================================================
"""



print("="*80)
print("GENERANDO REDES - SLEEP (AZUL)")
print("="*80)
print("\n⚠️ Esto tomará varios minutos...\n")

redes_sleep_dir = f'{RESULTS_PATH}/redes_sleep'
total_redes = 0

for sleep_name in data['sleep'].keys():

    subdir = f'{redes_sleep_dir}/{sleep_name}'
    num_ventanas = len(grafos_sleep[sleep_name])

    print(f"🔵 {sleep_name.upper()} ({num_ventanas} ventanas):")

    for idx, G in enumerate(grafos_sleep[sleep_name]):

        visualizar_red_completa(G, f'{sleep_name.upper()} - Ventana {idx+1}',
                               f'red_{sleep_name}_v{idx+1:03d}', subdir, '#92C5DE')

        total_redes += 1

        if (idx + 1) % 5 == 0 or (idx + 1) == num_ventanas:
            print(f"   [{idx+1}/{num_ventanas}]")

print(f"\n📁 Guardado en: {redes_sleep_dir}")
print(f"\n✅ Total: {total_redes} redes Sleep generadas")

In [ ]:
"""
=============================================================================
CELDA 8e: RESUMEN
=============================================================================
"""

n_taichi = len(list(Path(f'{RESULTS_PATH}/redes_taichi').glob('*.png'))) if Path(f'{RESULTS_PATH}/redes_taichi').exists() else 0
n_yoga = len(list(Path(f'{RESULTS_PATH}/redes_yoga').glob('*.png'))) if Path(f'{RESULTS_PATH}/redes_yoga').exists() else 0
n_sleep = len(list(Path(f'{RESULTS_PATH}/redes_sleep').glob('*/*.png'))) if Path(f'{RESULTS_PATH}/redes_sleep').exists() else 0

print("="*80)
print("RESUMEN DE REDES GENERADAS")
print("="*80)
print(f"""
   🟠 Tai-Chi (naranja):  {n_taichi} redes
   🟢 Yoga (verde):       {n_yoga} redes
   🔵 Sleep (azul):       {n_sleep} redes
   ─────────────────────────────
   TOTAL:                 {n_taichi + n_yoga + n_sleep} redes

📁 Ubicación: {RESULTS_PATH}
""")
print("✅ COMPLETADO")

In [ ]:
"""
=============================================================================
CELDA 9: GALERÍA DE EJEMPLO - REDES DE VISIBILIDAD
=============================================================================
"""

print("\n" + "="*80)
print("MOSTRANDO EJEMPLOS DE REDES GENERADAS")
print("="*80)

# Mostrar ejemplos
ejemplos = [
    ('Meditator 1', f'{redes_taichi_dir}/red_taichi1.png'),
    ('Meditator 1', f'{redes_yoga_dir}/red_yoga1.png'),
    ('Control 1 - Window 1', f'{redes_sleep_dir}/sleep1/red_sleep1_v001.png'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (nombre, filepath) in enumerate(ejemplos):
    if os.path.exists(filepath):
        img = plt.imread(filepath)
        axes[i].imshow(img)
        axes[i].set_title(nombre, fontsize=12, fontweight='bold')
        axes[i].axis('off')
        print(f"   ✓ Mostrando: {nombre}")
    else:
        axes[i].text(0.5, 0.5, f'Archivo no encontrado:\n{nombre}',
                    ha='center', va='center', transform=axes[i].transAxes)
        axes[i].axis('off')

#plt.suptitle('Ejemplos de Redes de Visibilidad Generadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/galeria_ejemplo_redes.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Galería de ejemplo guardada")

In [ ]:
"""
=============================================================================
TABLA DE MÉTRICAS - MEDITADORES
=============================================================================
"""

# Crear DataFrame con métricas de meditadores
datos_metricas = []

for name, metrics in metricas_meditadores.items():
    grupo = 'TaiChi' if 'taichi' in name else 'Yoga'
    fila = {'Sujeto': name, 'Grupo': grupo}
    fila.update(metrics.to_dict())
    datos_metricas.append(fila)

df_metricas_med = pd.DataFrame(datos_metricas)

print("="*80)
print("MÉTRICAS DE RED - MEDITADORES")
print("="*80)
print()
print(df_metricas_med.to_string(index=False))
print()

# Guardar
df_metricas_med.to_csv(f'{RESULTS_PATH}/metricas_meditadores.csv', index=False)
print(f"✅ Guardado: {RESULTS_PATH}/metricas_meditadores.csv")

# Estadísticas por grupo
print("\n" + "-"*80)
print("ESTADÍSTICAS POR GRUPO (MEDITADORES)")
print("-"*80)
print(df_metricas_med.groupby('Grupo').agg({
    'Densidad': ['mean', 'std'],
    'Grado_Promedio': ['mean', 'std'],
    'Clustering_Promedio': ['mean', 'std'],
    'Longitud_Camino_Promedio': ['mean', 'std']
}).round(4))

In [ ]:
"""
=============================================================================
COMPARACIONES EN PARES: CADA MEDITADOR VS CADA VENTANA DE SLEEP
=============================================================================
"""

print("🔄 Realizando comparaciones en pares...")
print("="*60)

# Lista para almacenar todas las comparaciones
todas_comparaciones = []

# Total de comparaciones
num_meditadores = len(metricas_meditadores)
num_ventanas_total = sum(len(metricas_sleep[s]) for s in metricas_sleep)
total_comparaciones = num_meditadores * num_ventanas_total

print(f"   Meditadores: {num_meditadores}")
print(f"   Ventanas Sleep totales: {num_ventanas_total}")
print(f"   Comparaciones a realizar: {total_comparaciones}")
print("-"*60)

contador = 0
for med_name, med_metrics in metricas_meditadores.items():
    grupo_med = 'TaiChi' if 'taichi' in med_name else 'Yoga'
    med_dict = med_metrics.to_dict()

    for sleep_name, sleep_metrics_list in metricas_sleep.items():
        for ventana_idx, sleep_metrics in enumerate(sleep_metrics_list):
            sleep_dict = sleep_metrics.to_dict()

            # Crear registro de comparación
            comparacion = {
                'Meditador': med_name,
                'Grupo_Meditador': grupo_med,
                'Control': sleep_name,
                'Ventana': ventana_idx + 1,
            }

            # Añadir métricas del meditador
            for key, value in med_dict.items():
                comparacion[f'Med_{key}'] = value

            # Añadir métricas del control
            for key, value in sleep_dict.items():
                comparacion[f'Ctrl_{key}'] = value

            # Calcular diferencias
            for key in med_dict.keys():
                diff = med_dict[key] - sleep_dict[key]
                comparacion[f'Diff_{key}'] = diff

                # Diferencia relativa (%)
                if sleep_dict[key] != 0:
                    comparacion[f'DiffRel_{key}'] = (diff / abs(sleep_dict[key])) * 100
                else:
                    comparacion[f'DiffRel_{key}'] = np.nan

            todas_comparaciones.append(comparacion)
            contador += 1

    print(f"   ✓ {med_name} comparado con todas las ventanas ({contador}/{total_comparaciones})")

# Crear DataFrame
df_comparaciones = pd.DataFrame(todas_comparaciones)

print("\n" + "="*60)
print(f"✅ Comparaciones completadas: {len(df_comparaciones)} registros")
print("="*60)

# Guardar
df_comparaciones.to_csv(f'{RESULTS_PATH}/comparaciones_completas.csv', index=False)
print(f"✅ Guardado: {RESULTS_PATH}/comparaciones_completas.csv")

In [ ]:
"""
=============================================================================
RESUMEN DE COMPARACIONES POR MEDITADOR
=============================================================================
"""

# Crear resumen por meditador
resumen_por_meditador = df_comparaciones.groupby(['Meditador', 'Grupo_Meditador']).agg({
    'Med_Densidad': 'first',
    'Med_Grado_Promedio': 'first',
    'Med_Clustering_Promedio': 'first',
    'Med_Longitud_Camino_Promedio': 'first',
    'Ctrl_Densidad': ['mean', 'std'],
    'Ctrl_Grado_Promedio': ['mean', 'std'],
    'Ctrl_Clustering_Promedio': ['mean', 'std'],
    'Diff_Densidad': ['mean', 'std', 'min', 'max'],
    'Diff_Grado_Promedio': ['mean', 'std', 'min', 'max'],
    'Diff_Clustering_Promedio': ['mean', 'std', 'min', 'max'],
}).round(4)

print("="*100)
print("RESUMEN DE COMPARACIONES POR MEDITADOR")
print("="*100)
print()
print(resumen_por_meditador)
print()

# Guardar resumen
resumen_por_meditador.to_csv(f'{RESULTS_PATH}/resumen_por_meditador.csv')
print(f"✅ Guardado: {RESULTS_PATH}/resumen_por_meditador.csv")

In [ ]:
"""
=============================================================================
CELDA 16: VISUALIZACIÓN - DIFERENCIAS PROMEDIO POR MEDITADOR
=============================================================================
"""

COLOR_TAICHI = '#F4A59B'
COLOR_YOGA = '#A1D99B'


metricas_visualizar = ['Densidad', 'Grado_Promedio', 'Clustering_Promedio',
                       'Longitud_Camino_Promedio', 'Asortatividad', 'Transitividad']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, metrica in enumerate(metricas_visualizar):
    ax = axes[i]

    diff_col = f'Diff_{metrica}'

    medias = df_comparaciones.groupby(['Meditador', 'Grupo_Meditador'])[diff_col].mean().reset_index()
    stds = df_comparaciones.groupby(['Meditador', 'Grupo_Meditador'])[diff_col].std().reset_index()

    x_pos = np.arange(len(medias))
    bar_colors = [COLOR_TAICHI if 'TaiChi' in g else COLOR_YOGA for g in medias['Grupo_Meditador']]

    bars = ax.bar(x_pos, medias[diff_col], yerr=stds[diff_col],
                  color=bar_colors, alpha=0.7, capsize=3, edgecolor='black')

    ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(medias['Meditador'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(f'Diferencia en {metrica}')
    ax.set_title(f'Δ {metrica.replace("_", " ")}')
    ax.grid(True, alpha=0.3, axis='y')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=COLOR_TAICHI, label='Tai-Chi'),
                   Patch(facecolor=COLOR_YOGA, label='Yoga')]
fig.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.suptitle('Diferencias Promedio: Meditador - Control (Sleep)\n(Barras de error = desviación estándar)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/05_diferencias_por_meditador.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/05_diferencias_por_meditador.png")

In [ ]:
"""
=============================================================================
CELDA 17: DISTRIBUCIÓN DE MÉTRICAS POR GRUPO (BOX PLOTS)
=============================================================================
"""

COLOR_TAICHI = '#F4A59B'  # Naranja
COLOR_YOGA = '#A1D99B'    # Verde
COLOR_SLEEP = '#92C5DE'   # Azul

metricas_box = ['Densidad', 'Grado_Promedio', 'Clustering_Promedio',
                'Longitud_Camino_Promedio', 'Asortatividad', 'Transitividad']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, metrica in enumerate(metricas_box):
    ax = axes[i]

    data_box = []
    labels = []

    # TaiChi
    taichi_vals = df_comparaciones[df_comparaciones['Grupo_Meditador'] == 'TaiChi'][f'Med_{metrica}'].unique()
    if len(taichi_vals) > 0:
        data_box.append(taichi_vals)
        labels.append('Tai-Chi')

    # Yoga
    yoga_vals = df_comparaciones[df_comparaciones['Grupo_Meditador'] == 'Yoga'][f'Med_{metrica}'].unique()
    if len(yoga_vals) > 0:
        data_box.append(yoga_vals)
        labels.append('Yoga')

    # Sleep
    sleep_vals = df_comparaciones[f'Ctrl_{metrica}'].values[::10]
    if len(sleep_vals) > 0:
        data_box.append(sleep_vals)
        labels.append('Sleep')

    bp = ax.boxplot(data_box, labels=labels, patch_artist=True)

    colors_box = [COLOR_TAICHI, COLOR_YOGA, COLOR_SLEEP]
    for patch, color in zip(bp['boxes'], colors_box[:len(data_box)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_ylabel(metrica.replace('_', ' '))
    ax.set_title(f'{metrica.replace("_", " ")}')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Distribución de Métricas de Red por Grupo\n🟠 Tai-Chi  🟢 Yoga  🔵 Sleep',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/06_boxplots_metricas.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/06_boxplots_metricas.png")

In [ ]:
"""
=============================================================================
CELDA 18: MANN-WHITNEY U - MEDITADORES VS SLEEP (2 GRUPOS)
=============================================================================
"""

print("="*80)
print("MANN-WHITNEY U: MEDITADORES (TAICHI + YOGA) VS SLEEP")
print("="*80)
print("\nNota: TaiChi y Yoga se combinan como un solo grupo 'Meditadores'")
print("-"*80)

metricas_test = ['Densidad', 'Grado_Promedio', 'Clustering_Promedio',
                 'Longitud_Camino_Promedio', 'Asortatividad', 'Transitividad',
                 'Betweenness_Promedio', 'Closeness_Promedio']

resultados_mann = []

for metrica in metricas_test:
    med_col = f'Med_{metrica}'
    ctrl_col = f'Ctrl_{metrica}'

    if med_col not in df_comparaciones.columns:
        continue

    # Combinar TaiChi + Yoga
    taichi_vals = df_comparaciones[df_comparaciones['Grupo_Meditador'] == 'TaiChi'][med_col].unique()
    yoga_vals = df_comparaciones[df_comparaciones['Grupo_Meditador'] == 'Yoga'][med_col].unique()
    meditadores_vals = np.concatenate([taichi_vals, yoga_vals])

    # Sleep (muestra)
    sleep_vals = df_comparaciones[ctrl_col].values[::20]

    if len(meditadores_vals) < 2 or len(sleep_vals) < 2:
        continue

    # Mann-Whitney U
    u_stat, p_mann = stats.mannwhitneyu(meditadores_vals, sleep_vals, alternative='two-sided')

    # Tamaño del efecto (r)
    n1, n2 = len(meditadores_vals), len(sleep_vals)
    mu_U = (n1 * n2) / 2
    sigma_U = np.sqrt((n1 * n2 * (n1 + n2 + 1)) / 12)
    z_score = (u_stat - mu_U) / sigma_U
    r_effect = abs(z_score) / np.sqrt(n1 + n2)

    # Dirección
    direccion = "Med > Sleep" if np.median(meditadores_vals) > np.median(sleep_vals) else "Sleep > Med"

    resultados_mann.append({
        'Metrica': metrica,
        'Mediana_Meditadores': np.median(meditadores_vals),
        'Mediana_Sleep': np.median(sleep_vals),
        'U_Statistic': u_stat,
        'P_Value': p_mann,
        'r_Effect': r_effect,
        'Direccion': direccion,
        'Significativo': '✓' if p_mann < 0.05 else '✗'
    })

df_mann = pd.DataFrame(resultados_mann)

# Mostrar
print(f"\n{'Métrica':<30} {'Med_Med':<10} {'Med_Slp':<10} {'p-valor':<10} {'r':<8} {'Sig':<5} {'Dirección'}")
print("-"*95)
for _, row in df_mann.iterrows():
    print(f"{row['Metrica']:<30} {row['Mediana_Meditadores']:<10.4f} {row['Mediana_Sleep']:<10.4f} "
          f"{row['P_Value']:<10.4f} {row['r_Effect']:<8.3f} {row['Significativo']:<5} {row['Direccion']}")

print("\n" + "-"*80)
print("Interpretación r: <0.1 Insignif. | 0.1-0.3 Pequeño | 0.3-0.5 Mediano | >0.5 Grande")

# Guardar
df_mann.to_csv(f'{RESULTS_PATH}/mann_whitney_2grupos.csv', index=False)
print(f"\n✅ Guardado: {RESULTS_PATH}/mann_whitney_2grupos.csv")

In [ ]:
"""
=============================================================================
CELDA 19: KRUSKAL-WALLIS + PRUEBA POST-HOC DE DUNN (CORRECCIÓN BONFERRONI)
=============================================================================
"""

# Intentar importar scikit-posthocs
try:
    import scikit_posthocs as sp
    HAS_POSTHOCS = True
    print("✅ scikit-posthocs disponible")
except ImportError:
    HAS_POSTHOCS = False
    print("⚠️ scikit-posthocs no instalado. Se usará implementación manual.")
    print("   Para instalar: pip install scikit-posthocs")


def dunn_test_manual(grupos, nombres_grupos):
    """
    Implementación manual de la prueba de Dunn con corrección de Bonferroni.
    """
    from itertools import combinations

    # Combinar todos los datos y calcular rangos
    todos_datos = np.concatenate(grupos)
    N = len(todos_datos)
    rangos = stats.rankdata(todos_datos, method='average')

    # Asignar rangos a cada grupo
    rangos_por_grupo = []
    idx = 0
    for grupo in grupos:
        n = len(grupo)
        rangos_por_grupo.append(rangos[idx:idx+n])
        idx += n

    # Calcular rango promedio de cada grupo
    rangos_promedio = [np.mean(r) for r in rangos_por_grupo]
    tamaños = [len(g) for g in grupos]

    # Número de comparaciones
    k = len(grupos)
    num_comparaciones = k * (k - 1) // 2

    resultados = []

    for (i, nombre_i), (j, nombre_j) in combinations(enumerate(nombres_grupos), 2):
        diff_rangos = abs(rangos_promedio[i] - rangos_promedio[j])
        se = np.sqrt((N * (N + 1) / 12) * (1/tamaños[i] + 1/tamaños[j]))
        z = diff_rangos / se
        p_valor = 2 * (1 - stats.norm.cdf(abs(z)))
        p_ajustado = min(p_valor * num_comparaciones, 1.0)

        resultados.append({
            'Grupo_1': nombre_i,
            'Grupo_2': nombre_j,
            'Rango_Prom_1': rangos_promedio[i],
            'Rango_Prom_2': rangos_promedio[j],
            'Z_Statistic': z,
            'P_Value': p_valor,
            'P_Value_Bonferroni': p_ajustado,
            'Significativo_005': '✓' if p_ajustado < 0.05 else '✗',
            'Significativo_001': '✓' if p_ajustado < 0.01 else '✗'
        })

    return pd.DataFrame(resultados)


print("\n" + "="*80)
print("KRUSKAL-WALLIS + POST-HOC DE DUNN (CORRECCIÓN BONFERRONI)")
print("="*80)
print("\n📋 Metodología:")
print("   1. Kruskal-Wallis: Prueba ómnibus para diferencias globales")
print("   2. Dunn Post-Hoc: Comparaciones por pares (si K-W es significativo)")
print("   3. Corrección Bonferroni: α/3 = 0.0167")
print("-"*80)

metricas_test = ['Densidad', 'Grado_Promedio', 'Clustering_Promedio',
                 'Longitud_Camino_Promedio', 'Asortatividad', 'Transitividad',
                 'Betweenness_Promedio', 'Closeness_Promedio']

resultados_kruskal = []
resultados_dunn_todos = []

for metrica in metricas_test:
    med_col = f'Med_{metrica}'
    ctrl_col = f'Ctrl_{metrica}'

    if med_col not in df_comparaciones.columns:
        continue

    # Obtener datos por grupo
    taichi_vals = df_comparaciones[df_comparaciones['Grupo_Meditador'] == 'TaiChi'][med_col].unique()
    yoga_vals = df_comparaciones[df_comparaciones['Grupo_Meditador'] == 'Yoga'][med_col].unique()
    sleep_vals = df_comparaciones[ctrl_col].values[::50]

    if len(taichi_vals) < 2 or len(yoga_vals) < 2 or len(sleep_vals) < 2:
        continue

    # KRUSKAL-WALLIS
    h_stat, p_kruskal = stats.kruskal(taichi_vals, yoga_vals, sleep_vals)

    n_total = len(taichi_vals) + len(yoga_vals) + len(sleep_vals)
    eta_sq = (h_stat - 2) / (n_total - 3) if n_total > 3 else 0

    resultados_kruskal.append({
        'Metrica': metrica,
        'N_TaiChi': len(taichi_vals),
        'N_Yoga': len(yoga_vals),
        'N_Sleep': len(sleep_vals),
        'Mediana_TaiChi': np.median(taichi_vals),
        'Mediana_Yoga': np.median(yoga_vals),
        'Mediana_Sleep': np.median(sleep_vals),
        'H_Statistic': h_stat,
        'P_Value_KW': p_kruskal,
        'Eta_Squared': eta_sq,
        'Significativo_005': '✓' if p_kruskal < 0.05 else '✗'
    })

    # DUNN POST-HOC
    grupos = [taichi_vals, yoga_vals, sleep_vals]
    nombres = ['TaiChi', 'Yoga', 'Sleep']

    if HAS_POSTHOCS:
        data_dunn = []
        for vals, nombre in zip(grupos, nombres):
            for v in vals:
                data_dunn.append({'valor': v, 'grupo': nombre})
        df_dunn_temp = pd.DataFrame(data_dunn)

        dunn_matrix = sp.posthoc_dunn(df_dunn_temp, val_col='valor', group_col='grupo',
                                       p_adjust='bonferroni')

        for i, g1 in enumerate(nombres):
            for j, g2 in enumerate(nombres):
                if i < j:
                    p_adj = dunn_matrix.loc[g1, g2]
                    resultados_dunn_todos.append({
                        'Metrica': metrica,
                        'Grupo_1': g1,
                        'Grupo_2': g2,
                        'P_Value_Dunn_Bonferroni': p_adj,
                        'Significativo_005': '✓' if p_adj < 0.05 else '✗',
                        'KW_Significativo': '✓' if p_kruskal < 0.05 else '✗'
                    })
    else:
        df_dunn_manual = dunn_test_manual(grupos, nombres)
        for _, row in df_dunn_manual.iterrows():
            resultados_dunn_todos.append({
                'Metrica': metrica,
                'Grupo_1': row['Grupo_1'],
                'Grupo_2': row['Grupo_2'],
                'Z_Statistic': row['Z_Statistic'],
                'P_Value_Dunn_Bonferroni': row['P_Value_Bonferroni'],
                'Significativo_005': row['Significativo_005'],
                'KW_Significativo': '✓' if p_kruskal < 0.05 else '✗'
            })

df_kruskal = pd.DataFrame(resultados_kruskal)
df_dunn = pd.DataFrame(resultados_dunn_todos)

# MOSTRAR RESULTADOS
print("\n" + "="*80)
print("RESULTADOS KRUSKAL-WALLIS")
print("="*80)
print(df_kruskal[['Metrica', 'Mediana_TaiChi', 'Mediana_Yoga', 'Mediana_Sleep',
                  'H_Statistic', 'P_Value_KW', 'Significativo_005']].to_string(index=False))

print("\n" + "="*80)
print("RESULTADOS DUNN POST-HOC (BONFERRONI)")
print("="*80)
for metrica in df_dunn['Metrica'].unique():
    df_met = df_dunn[df_dunn['Metrica'] == metrica]
    print(f"\n{metrica}:")
    for _, row in df_met.iterrows():
        print(f"   {row['Grupo_1']} vs {row['Grupo_2']}: p_adj = {row['P_Value_Dunn_Bonferroni']:.4f} {row['Significativo_005']}")

# Guardar
df_kruskal.to_csv(f'{RESULTS_PATH}/kruskal_wallis_resultados.csv', index=False)
df_dunn.to_csv(f'{RESULTS_PATH}/dunn_posthoc_bonferroni.csv', index=False)
print(f"\n✅ Guardado: {RESULTS_PATH}/kruskal_wallis_resultados.csv")
print(f"✅ Guardado: {RESULTS_PATH}/dunn_posthoc_bonferroni.csv")

In [ ]:
"""
=============================================================================
CELDA 20: VISUALIZACIÓN KRUSKAL-WALLIS + DUNN POST-HOC
=============================================================================
"""

COLOR_TAICHI = '#F4A59B'  # Naranja
COLOR_YOGA = '#A1D99B'    # Verde
COLOR_SLEEP = '#92C5DE'   # Azul

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# ===== Gráfico 1: P-values Kruskal-Wallis =====
ax1 = axes[0, 0]
metricas_plot = df_kruskal['Metrica'].values
p_values_kw = df_kruskal['P_Value_KW'].values
colors_kw = ['#27ae60' if p < 0.05 else '#e74c3c' for p in p_values_kw]

y_pos = np.arange(len(metricas_plot))
ax1.barh(y_pos, -np.log10(p_values_kw), color=colors_kw, alpha=0.7, edgecolor='black')
ax1.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=2, label='α = 0.05')
ax1.axvline(x=-np.log10(0.01), color='orange', linestyle='--', linewidth=2, label='α = 0.01')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(metricas_plot, fontsize=9)
ax1.set_xlabel('-log₁₀(p-value)')
ax1.set_title('Kruskal-Wallis: Significancia Global\n(verde = p < 0.05)', fontsize=11)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3, axis='x')

# ===== Gráfico 2: Tamaño del efecto =====
ax2 = axes[0, 1]
eta_vals = df_kruskal['Eta_Squared'].values
colors_eta = ['#27ae60' if e >= 0.14 else '#f39c12' if e >= 0.06 else '#e74c3c' for e in eta_vals]

ax2.barh(y_pos, eta_vals, color=colors_eta, alpha=0.7, edgecolor='black')
ax2.axvline(x=0.01, color='#e74c3c', linestyle='--', alpha=0.7, label='Pequeño')
ax2.axvline(x=0.06, color='#f39c12', linestyle='--', alpha=0.7, label='Mediano')
ax2.axvline(x=0.14, color='#27ae60', linestyle='--', alpha=0.7, label='Grande')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(metricas_plot, fontsize=9)
ax2.set_xlabel('η² (Eta cuadrado)')
ax2.set_title('Tamaño del Efecto', fontsize=11)
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(True, alpha=0.3, axis='x')

# ===== Gráfico 3: Heatmap Dunn Post-Hoc =====
ax3 = axes[1, 0]
comparaciones = ['TaiChi vs Yoga', 'TaiChi vs Sleep', 'Yoga vs Sleep']
matriz_p = np.zeros((len(metricas_plot), 3))

for i, metrica in enumerate(metricas_plot):
    df_met = df_dunn[df_dunn['Metrica'] == metrica]
    for j, (g1, g2) in enumerate([('TaiChi', 'Yoga'), ('TaiChi', 'Sleep'), ('Yoga', 'Sleep')]):
        row = df_met[((df_met['Grupo_1'] == g1) & (df_met['Grupo_2'] == g2)) |
                     ((df_met['Grupo_1'] == g2) & (df_met['Grupo_2'] == g1))]
        if len(row) > 0:
            matriz_p[i, j] = row['P_Value_Dunn_Bonferroni'].values[0]

im = ax3.imshow(matriz_p, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.1)
ax3.set_xticks(np.arange(3))
ax3.set_yticks(np.arange(len(metricas_plot)))
ax3.set_xticklabels(comparaciones, fontsize=9, rotation=15, ha='right')
ax3.set_yticklabels(metricas_plot, fontsize=9)

for i in range(len(metricas_plot)):
    for j in range(3):
        p = matriz_p[i, j]
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        color = 'white' if p < 0.05 else 'black'
        ax3.text(j, i, f'{p:.3f}{sig}', ha='center', va='center', fontsize=8, color=color)

ax3.set_title('Dunn Post-Hoc (Bonferroni)\n* p<0.05, ** p<0.01, *** p<0.001', fontsize=11)
plt.colorbar(im, ax=ax3, label='p-valor', shrink=0.8)

# ===== Gráfico 4: Resumen de significancias =====
ax4 = axes[1, 1]
sig_counts = {'TaiChi vs Yoga': 0, 'TaiChi vs Sleep': 0, 'Yoga vs Sleep': 0}

for _, row in df_dunn.iterrows():
    if row['Significativo_005'] == '✓':
        key = f"{row['Grupo_1']} vs {row['Grupo_2']}"
        if key in sig_counts:
            sig_counts[key] += 1

x_pos = np.arange(3)
colors_bar = [COLOR_TAICHI, COLOR_YOGA, COLOR_SLEEP]  # Usar colores del grupo principal
bars = ax4.bar(x_pos, list(sig_counts.values()), color=colors_bar, alpha=0.7, edgecolor='black')

for bar, val in zip(bars, sig_counts.values()):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val}/{len(metricas_plot)}', ha='center', fontsize=11, fontweight='bold')

ax4.set_xticks(x_pos)
ax4.set_xticklabels(list(sig_counts.keys()), fontsize=10)
ax4.set_ylabel('Métricas significativas')
ax4.set_title('Resumen de Comparaciones\nSignificativas', fontsize=11)
ax4.set_ylim(0, len(metricas_plot) + 1)
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('ANÁLISIS KRUSKAL-WALLIS + DUNN POST-HOC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/07_kruskal_dunn_completo.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/07_kruskal_dunn_completo.png")

In [ ]:
"""
=============================================================================
CELDA 20b: TABLA RESUMEN E INTERPRETACIÓN
=============================================================================
"""

print("\n" + "="*80)
print("                    RESUMEN E INTERPRETACIÓN")
print("="*80)

print(f"\n{'Métrica':<28} {'K-W p':<10} {'Sig':<6} {'TC vs Y':<10} {'TC vs S':<10} {'Y vs S':<10}")
print("-"*80)

for metrica in df_kruskal['Metrica'].values:
    kw_row = df_kruskal[df_kruskal['Metrica'] == metrica].iloc[0]
    dunn_met = df_dunn[df_dunn['Metrica'] == metrica]

    def get_sig(g1, g2):
        row = dunn_met[((dunn_met['Grupo_1'] == g1) & (dunn_met['Grupo_2'] == g2)) |
                       ((dunn_met['Grupo_1'] == g2) & (dunn_met['Grupo_2'] == g1))]
        return row['Significativo_005'].values[0] if len(row) > 0 else '-'

    print(f"{metrica:<28} {kw_row['P_Value_KW']:<10.4f} {kw_row['Significativo_005']:<6} "
          f"{get_sig('TaiChi', 'Yoga'):<10} {get_sig('TaiChi', 'Sleep'):<10} {get_sig('Yoga', 'Sleep'):<10}")

print("-"*80)
print("✓ = Significativo (p < 0.05 con Bonferroni)  |  ✗ = No significativo")

# Conclusiones
print("\n" + "="*80)
print("CONCLUSIONES")
print("="*80)

kw_sig = df_kruskal[df_kruskal['Significativo_005'] == '✓']['Metrica'].tolist()
print(f"\n📊 Métricas con diferencias globales (K-W p < 0.05): {len(kw_sig)}")
for m in kw_sig:
    print(f"   • {m}")

print("\n📊 Diferencias específicas (Dunn Post-Hoc):")
for comp in [('TaiChi', 'Yoga'), ('TaiChi', 'Sleep'), ('Yoga', 'Sleep')]:
    g1, g2 = comp
    sig = df_dunn[((df_dunn['Grupo_1'] == g1) & (df_dunn['Grupo_2'] == g2) & (df_dunn['Significativo_005'] == '✓')) |
                  ((df_dunn['Grupo_1'] == g2) & (df_dunn['Grupo_2'] == g1) & (df_dunn['Significativo_005'] == '✓'))]['Metrica'].tolist()
    print(f"\n   {g1} vs {g2}: {len(sig)} métricas")
    for m in sig:
        print(f"      • {m}")

In [ ]:
"""
=============================================================================
CELDA 21: HEATMAP DE DIFERENCIAS PROMEDIO
=============================================================================
"""

metricas_heatmap = ['Densidad', 'Grado_Promedio', 'Clustering_Promedio',
                    'Longitud_Camino_Promedio', 'Asortatividad', 'Transitividad']

diff_cols = [f'Diff_{m}' for m in metricas_heatmap]
pivot_data = df_comparaciones.groupby('Meditador')[diff_cols].mean()
pivot_data.columns = [c.replace('Diff_', '').replace('_', '\n') for c in pivot_data.columns]

orden = ['taichi1', 'taichi2', 'taichi3', 'taichi4', 'yoga1', 'yoga2', 'yoga3', 'yoga4']
pivot_data = pivot_data.reindex([o for o in orden if o in pivot_data.index])

fig, ax = plt.subplots(figsize=(12, 8))

im = ax.imshow(pivot_data.values, cmap='RdBu_r', aspect='auto',
               vmin=-np.abs(pivot_data.values).max(),
               vmax=np.abs(pivot_data.values).max())

ax.set_xticks(np.arange(len(pivot_data.columns)))
ax.set_yticks(np.arange(len(pivot_data.index)))
ax.set_xticklabels(pivot_data.columns, fontsize=9)
ax.set_yticklabels(pivot_data.index, fontsize=10)

for i in range(len(pivot_data.index)):
    for j in range(len(pivot_data.columns)):
        valor = pivot_data.values[i, j]
        color = 'white' if abs(valor) > np.abs(pivot_data.values).max() * 0.5 else 'black'
        ax.text(j, i, f'{valor:.4f}', ha='center', va='center', fontsize=9, color=color)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Diferencia (Meditador - Sleep)', fontsize=10)

# Línea separadora entre TaiChi y Yoga
ax.axhline(y=3.5, color='black', linewidth=3)

# Etiquetas de grupo con colores
ax.text(-0.8, 1.5, 'TAI-CHI 🟠', fontsize=11, fontweight='bold', rotation=90, va='center', color='#e67e22')
ax.text(-0.8, 5.5, 'YOGA 🟢', fontsize=11, fontweight='bold', rotation=90, va='center', color='#2ecc71')

ax.set_title('Heatmap de Diferencias Promedio en Métricas de Red\n(Positivo = Meditador > Sleep 🔵)',
             fontsize=13, fontweight='bold', pad=20)
ax.set_xlabel('Métrica de Red', fontsize=11)
ax.set_ylabel('Sujeto', fontsize=11)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/08_heatmap_diferencias.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/08_heatmap_diferencias.png")

In [ ]:
"""
=============================================================================
CELDA 22: COMPARACIÓN VISUAL DETALLADA - EJEMPLO
=============================================================================
"""

COLOR_TAICHI = '#F4A59B'
COLOR_SLEEP = '#92C5DE'

med_ejemplo = 'taichi1'
sleep_ejemplo = 'sleep1'
ventana_ejemplo = 0

med_signal = data['taichi'][med_ejemplo]
sleep_signal = data['sleep'][sleep_ejemplo]['segments'][ventana_ejemplo]

G_med = grafos_meditadores[med_ejemplo]
G_sleep = grafos_sleep[sleep_ejemplo][ventana_ejemplo]

metrics_med = metricas_meditadores[med_ejemplo]
metrics_sleep = metricas_sleep[sleep_ejemplo][ventana_ejemplo]

fig = plt.figure(figsize=(16, 14))
gs = GridSpec(3, 2, figure=fig, height_ratios=[1, 1, 0.6])

# ===== MEDITADOR - Señal + Red =====
ax1 = fig.add_subplot(gs[0, 0])
x_med = np.arange(len(med_signal))
for edge in list(G_med.edges())[:400]:
    i, j = edge
    ax1.plot([x_med[i], x_med[j]], [med_signal[i], med_signal[j]],
             'gray', alpha=0.2, linewidth=0.3)
ax1.plot(x_med, med_signal, color=COLOR_TAICHI, linewidth=1, zorder=5)
ax1.scatter(x_med[::10], med_signal[::10], c=COLOR_TAICHI, s=10, zorder=6)
ax1.set_title(f'{med_ejemplo.upper()} 🟠 - Señal ECG + Red de Visibilidad', fontsize=12)
ax1.set_xlabel('Tiempo (muestras)')
ax1.set_ylabel('Amplitud')
ax1.grid(True, alpha=0.3)

# ===== SLEEP - Señal + Red =====
ax2 = fig.add_subplot(gs[0, 1])
x_sleep = np.arange(len(sleep_signal))
for edge in list(G_sleep.edges())[:400]:
    i, j = edge
    ax2.plot([x_sleep[i], x_sleep[j]], [sleep_signal[i], sleep_signal[j]],
             'gray', alpha=0.2, linewidth=0.3)
ax2.plot(x_sleep, sleep_signal, color=COLOR_SLEEP, linewidth=1, zorder=5)
ax2.scatter(x_sleep[::10], sleep_signal[::10], c=COLOR_SLEEP, s=10, zorder=6)
ax2.set_title(f'{sleep_ejemplo.upper()} (Ventana {ventana_ejemplo+1}) 🔵 - Señal ECG + Red', fontsize=12)
ax2.set_xlabel('Tiempo (muestras)')
ax2.set_ylabel('Amplitud')
ax2.grid(True, alpha=0.3)

# ===== MEDITADOR - Topología =====
ax3 = fig.add_subplot(gs[1, 0])
nodes_sub = list(range(0, G_med.number_of_nodes(), max(1, G_med.number_of_nodes()//80)))
G_med_sub = G_med.subgraph(nodes_sub)
pos_med = nx.spring_layout(G_med_sub, seed=42)
nx.draw(G_med_sub, pos_med, ax=ax3, node_size=50, node_color=COLOR_TAICHI,
        edge_color='gray', alpha=0.7, width=0.5, with_labels=False)
ax3.set_title(f'Topología - {med_ejemplo.upper()} 🟠\nNodos: {metrics_med.num_nodes} | Aristas: {metrics_med.num_edges}', fontsize=11)

# ===== SLEEP - Topología =====
ax4 = fig.add_subplot(gs[1, 1])
nodes_sub_s = list(range(0, G_sleep.number_of_nodes(), max(1, G_sleep.number_of_nodes()//80)))
G_sleep_sub = G_sleep.subgraph(nodes_sub_s)
pos_sleep = nx.spring_layout(G_sleep_sub, seed=42)
nx.draw(G_sleep_sub, pos_sleep, ax=ax4, node_size=50, node_color=COLOR_SLEEP,
        edge_color='gray', alpha=0.7, width=0.5, with_labels=False)
ax4.set_title(f'Topología - {sleep_ejemplo.upper()} V{ventana_ejemplo+1} 🔵\nNodos: {metrics_sleep.num_nodes} | Aristas: {metrics_sleep.num_edges}', fontsize=11)

# ===== TABLA COMPARATIVA =====
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')

table_data = [
    ['MÉTRICA', f'{med_ejemplo.upper()} 🟠', f'{sleep_ejemplo.upper()} V{ventana_ejemplo+1} 🔵', 'DIFERENCIA'],
    ['Nodos', f'{metrics_med.num_nodes}', f'{metrics_sleep.num_nodes}',
     f'{metrics_med.num_nodes - metrics_sleep.num_nodes:+d}'],
    ['Aristas', f'{metrics_med.num_edges}', f'{metrics_sleep.num_edges}',
     f'{metrics_med.num_edges - metrics_sleep.num_edges:+d}'],
    ['Densidad', f'{metrics_med.density:.4f}', f'{metrics_sleep.density:.4f}',
     f'{metrics_med.density - metrics_sleep.density:+.4f}'],
    ['Grado Promedio', f'{metrics_med.avg_degree:.2f}', f'{metrics_sleep.avg_degree:.2f}',
     f'{metrics_med.avg_degree - metrics_sleep.avg_degree:+.2f}'],
    ['Clustering', f'{metrics_med.avg_clustering:.4f}', f'{metrics_sleep.avg_clustering:.4f}',
     f'{metrics_med.avg_clustering - metrics_sleep.avg_clustering:+.4f}'],
    ['Long. Camino', f'{metrics_med.avg_path_length:.2f}', f'{metrics_sleep.avg_path_length:.2f}',
     f'{metrics_med.avg_path_length - metrics_sleep.avg_path_length:+.2f}'],
    ['Asortatividad', f'{metrics_med.assortativity:.4f}', f'{metrics_sleep.assortativity:.4f}',
     f'{metrics_med.assortativity - metrics_sleep.assortativity:+.4f}'],
    ['Transitividad', f'{metrics_med.transitivity:.4f}', f'{metrics_sleep.transitivity:.4f}',
     f'{metrics_med.transitivity - metrics_sleep.transitivity:+.4f}'],
]

table = ax5.table(cellText=table_data, loc='center', cellLoc='center',
                  colWidths=[0.25, 0.2, 0.25, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.8)

for j in range(4):
    table[(0, j)].set_facecolor('#2c3e50')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

for i in range(1, len(table_data)):
    table[(i, 1)].set_facecolor('#fdebd0')  # Naranja claro
    table[(i, 2)].set_facecolor('#d6eaf8')  # Azul claro

plt.suptitle('COMPARACIÓN DETALLADA: Meditador vs Control', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/09_comparacion_detallada.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/09_comparacion_detallada.png")

In [ ]:
"""
=============================================================================
CELDA 23: GENERAR COMPARACIONES INDIVIDUALES PARA CADA MEDITADOR
=============================================================================
"""

COLOR_TAICHI = '#F4A59B'  # Naranja
COLOR_YOGA = '#A1D99B'    # Verde
COLOR_SLEEP = '#92C5DE'   # Azul

print("="*80)
print("GENERANDO COMPARACIONES INDIVIDUALES")
print("="*80)

comparaciones_path = f'{RESULTS_PATH}/comparaciones_individuales'
os.makedirs(comparaciones_path, exist_ok=True)

todos_meditadores = list(data['taichi'].keys()) + list(data['yoga'].keys())

for med_name in todos_meditadores:
    if 'taichi' in med_name:
        med_signal = data['taichi'][med_name]
        color_med = COLOR_TAICHI
        grupo = 'Tai-Chi 🟠'
    else:
        med_signal = data['yoga'][med_name]
        color_med = COLOR_YOGA
        grupo = 'Yoga 🟢'

    metrics_med = metricas_meditadores[med_name]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    sleep_sujetos = ['sleep1', 'sleep3', 'sleep5', 'sleep7']

    for col, sleep_name in enumerate(sleep_sujetos):
        sleep_signal = data['sleep'][sleep_name]['segments'][0]
        metrics_sleep_temp = metricas_sleep[sleep_name][0]

        # Fila superior: Señales
        ax_top = axes[0, col]
        ax_top.plot(med_signal, color=color_med, linewidth=0.8, alpha=0.7, label=med_name)
        ax_top.plot(sleep_signal, color=COLOR_SLEEP, linewidth=0.8, alpha=0.7, label=sleep_name)
        ax_top.set_title(f'{med_name} vs {sleep_name}', fontsize=10)
        ax_top.legend(fontsize=7, loc='upper right')
        ax_top.set_xlabel('Muestras', fontsize=8)
        ax_top.set_ylabel('Amplitud', fontsize=8)
        ax_top.tick_params(labelsize=7)
        ax_top.grid(True, alpha=0.3)

        # Fila inferior: Barras de métricas
        ax_bot = axes[1, col]
        metricas_nombres = ['Densidad', 'Clustering', 'Grado/100', 'L.Camino/10']
        valores_med = [metrics_med.density, metrics_med.avg_clustering,
                       metrics_med.avg_degree/100, metrics_med.avg_path_length/10]
        valores_sleep = [metrics_sleep_temp.density, metrics_sleep_temp.avg_clustering,
                        metrics_sleep_temp.avg_degree/100, metrics_sleep_temp.avg_path_length/10]

        x = np.arange(len(metricas_nombres))
        width = 0.35

        ax_bot.bar(x - width/2, valores_med, width, label=med_name, color=color_med, alpha=0.7)
        ax_bot.bar(x + width/2, valores_sleep, width, label=sleep_name, color=COLOR_SLEEP, alpha=0.7)
        ax_bot.set_xticks(x)
        ax_bot.set_xticklabels(metricas_nombres, fontsize=8, rotation=45, ha='right')
        ax_bot.legend(fontsize=7)
        ax_bot.set_ylabel('Valor', fontsize=8)
        ax_bot.tick_params(labelsize=7)
        ax_bot.grid(True, alpha=0.3, axis='y')

    plt.suptitle(f'Comparaciones: {med_name.upper()} ({grupo}) vs Grupo Sleep 🔵',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{comparaciones_path}/comp_{med_name}.png', dpi=120, bbox_inches='tight')
    plt.close()

    print(f"   ✓ {med_name}")

print(f"\n📁 Guardado en: {comparaciones_path}")
print("✅ Comparaciones individuales completadas")

In [ ]:
"""
=============================================================================
CELDA 24: DISTRIBUCIÓN DE GRADO POR GRUPO
=============================================================================
"""

COLOR_TAICHI = '#F4A59B'  # Naranja
COLOR_YOGA = '#A1D99B'    # Verde
COLOR_SLEEP = '#92C5DE'   # Azul

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Recopilar grados
grados_taichi = []
grados_yoga = []
grados_sleep = []

for name in data['taichi'].keys():
    G = grafos_meditadores[name]
    grados_taichi.extend([d for n, d in G.degree()])

for name in data['yoga'].keys():
    G = grafos_meditadores[name]
    grados_yoga.extend([d for n, d in G.degree()])

for name in data['sleep'].keys():
    for G in grafos_sleep[name][:5]:
        grados_sleep.extend([d for n, d in G.degree()])

# Gráficos
grupos_data = [
    ('Tai-Chi 🟠', grados_taichi, COLOR_TAICHI),
    ('Yoga 🟢', grados_yoga, COLOR_YOGA),
    ('Sleep 🔵', grados_sleep, COLOR_SLEEP)
]

for i, (nombre, grados, color) in enumerate(grupos_data):
    ax = axes[i]

    ax.hist(grados, bins=30, density=True, alpha=0.7, color=color, edgecolor='black', linewidth=0.5)

    media = np.mean(grados)
    mediana = np.median(grados)
    std = np.std(grados)

    ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Media: {media:.1f}')
    ax.axvline(mediana, color='orange', linestyle=':', linewidth=2, label=f'Mediana: {mediana:.1f}')

    ax.set_xlabel('Grado (k)', fontsize=11)
    ax.set_ylabel('P(k)', fontsize=11)
    ax.set_title(f'{nombre}\nn={len(grados)}, σ={std:.1f}', fontsize=12)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribución de Grado de las Redes de Visibilidad', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/10_distribucion_grado.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"✅ Figura guardada: {RESULTS_PATH}/10_distribucion_grado.png")

# Estadísticas
print("\n" + "="*60)
print("ESTADÍSTICAS DE DISTRIBUCIÓN DE GRADO")
print("="*60)
print(f"\n{'Grupo':<15} {'Media':<10} {'Mediana':<10} {'Std':<10} {'Min':<8} {'Max':<8}")
print("-"*60)
for nombre, grados, _ in grupos_data:
    print(f"{nombre:<15} {np.mean(grados):<10.2f} {np.median(grados):<10.2f} "
          f"{np.std(grados):<10.2f} {min(grados):<8} {max(grados):<8}")

In [ ]:
"""
=============================================================================
CELDA 25: RESUMEN FINAL DEL ANÁLISIS
=============================================================================
"""

print("\n" + "="*80)
print("                         RESUMEN FINAL DEL ANÁLISIS")
print("="*80)

# Contar archivos
archivos_csv = list(Path(RESULTS_PATH).glob('*.csv'))
archivos_png = list(Path(RESULTS_PATH).glob('*.png'))
comparaciones_ind = list(Path(f'{RESULTS_PATH}/comparaciones_individuales').glob('*.png')) if Path(f'{RESULTS_PATH}/comparaciones_individuales').exists() else []

print(f"""
📊 DATOS ANALIZADOS:
{'─'*50}
• Sujetos Tai-Chi: {len(data['taichi'])} (650 puntos cada uno)
• Sujetos Yoga: {len(data['yoga'])} (650 puntos cada uno)
• Sujetos Sleep: {len(data['sleep'])} ({data['sleep']['sleep1']['num_points']} puntos cada uno)
• Ventanas Sleep por sujeto: {len(data['sleep']['sleep1']['segments'])}
• Total de comparaciones realizadas: {len(df_comparaciones)}

🔬 REDES CONSTRUIDAS:
{'─'*50}
• Tipo de grafo: Visibilidad Natural (NVG)
• Grafos de meditadores: {len(grafos_meditadores)}
• Grafos de Sleep: {sum(len(v) for v in grafos_sleep.values())}

📈 PRUEBAS ESTADÍSTICAS REALIZADAS:
{'─'*50}
• Mann-Whitney U: Meditadores (combinados) vs Sleep
• Kruskal-Wallis: TaiChi vs Yoga vs Sleep (3 grupos)
• Dunn Post-Hoc: Comparaciones por pares con corrección Bonferroni
""")

# Resultados de Kruskal-Wallis
print("📋 RESULTADOS KRUSKAL-WALLIS (diferencias globales):")
print("─"*50)
kw_sig = df_kruskal[df_kruskal['Significativo_005'] == '✓']['Metrica'].tolist()
kw_no_sig = df_kruskal[df_kruskal['Significativo_005'] == '✗']['Metrica'].tolist()

if kw_sig:
    print("\n   ✅ Métricas SIGNIFICATIVAS (p < 0.05):")
    for m in kw_sig:
        p_val = df_kruskal[df_kruskal['Metrica'] == m]['P_Value_KW'].values[0]
        print(f"      • {m}: p = {p_val:.4f}")
else:
    print("\n   ⚠️ Ninguna métrica significativa")

if kw_no_sig:
    print("\n   ❌ Métricas NO significativas:")
    for m in kw_no_sig:
        print(f"      • {m}")

# Resultados de Dunn Post-Hoc
print("\n📋 RESULTADOS DUNN POST-HOC (comparaciones por pares):")
print("─"*50)

for comp in [('TaiChi', 'Yoga'), ('TaiChi', 'Sleep'), ('Yoga', 'Sleep')]:
    g1, g2 = comp
    sig_metricas = df_dunn[
        (((df_dunn['Grupo_1'] == g1) & (df_dunn['Grupo_2'] == g2)) |
         ((df_dunn['Grupo_1'] == g2) & (df_dunn['Grupo_2'] == g1))) &
        (df_dunn['Significativo_005'] == '✓')
    ]['Metrica'].tolist()

    print(f"\n   {g1} vs {g2}: {len(sig_metricas)} métricas significativas")
    if sig_metricas:
        for m in sig_metricas:
            print(f"      • {m}")
    else:
        print("      (ninguna)")

# Archivos generados
print(f"""

📁 ARCHIVOS GENERADOS:
{'─'*50}
   📄 CSV: {len(archivos_csv)} archivos
""")
for f in archivos_csv:
    print(f"      • {f.name}")

print(f"""
   📊 Figuras principales: {len(archivos_png)} archivos
""")
for f in archivos_png:
    print(f"      • {f.name}")

print(f"""
   📊 Comparaciones individuales: {len(comparaciones_ind)} archivos

📂 Ubicación de resultados: {RESULTS_PATH}
""")

print("="*80)
print("✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*80)

In [ ]:
"""
=============================================================================
CELDA 26: EXPORTAR REPORTE COMPLETO A EXCEL
=============================================================================
"""

try:
    excel_path = f'{RESULTS_PATH}/reporte_completo_redes_ecg.xlsx'

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

        # Hoja 1: Métricas de meditadores
        df_metricas_med.to_excel(writer, sheet_name='Metricas_Meditadores', index=False)

        # Hoja 2: Comparaciones (muestra de 1000 filas)
        df_comparaciones.head(1000).to_excel(writer, sheet_name='Comparaciones_Muestra', index=False)

        # Hoja 3: Mann-Whitney U (2 grupos)
        df_mann.to_excel(writer, sheet_name='MannWhitney_2grupos', index=False)

        # Hoja 4: Kruskal-Wallis (3 grupos)
        df_kruskal.to_excel(writer, sheet_name='KruskalWallis_3grupos', index=False)

        # Hoja 5: Dunn Post-Hoc
        df_dunn.to_excel(writer, sheet_name='Dunn_PostHoc_Bonferroni', index=False)

        # Hoja 6: Resumen por meditador
        resumen = df_comparaciones.groupby('Meditador').agg({
            'Med_Densidad': 'first',
            'Med_Grado_Promedio': 'first',
            'Med_Clustering_Promedio': 'first',
            'Diff_Densidad': ['mean', 'std'],
            'Diff_Grado_Promedio': ['mean', 'std'],
            'Diff_Clustering_Promedio': ['mean', 'std']
        }).round(4)
        resumen.to_excel(writer, sheet_name='Resumen_Por_Meditador')

    print(f"✅ Reporte Excel guardado exitosamente:")
    print(f"   📁 {excel_path}")
    print(f"\n   Hojas incluidas:")
    print(f"      • Metricas_Meditadores")
    print(f"      • Comparaciones_Muestra")
    print(f"      • MannWhitney_2grupos")
    print(f"      • KruskalWallis_3grupos")
    print(f"      • Dunn_PostHoc_Bonferroni")
    print(f"      • Resumen_Por_Meditador")

except ImportError:
    print("⚠️ openpyxl no está instalado.")
    print("   Para instalar, ejecuta: pip install openpyxl")
    print("   Los datos están disponibles en los archivos CSV.")

except Exception as e:
    print(f"⚠️ Error al exportar a Excel: {e}")
    print("   Los datos están disponibles en los archivos CSV.")

In [ ]:
"""
=============================================================================
CELDA 27: VERIFICACIÓN FINAL DE ARCHIVOS
=============================================================================
"""

print("🔍 VERIFICACIÓN DE ARCHIVOS GENERADOS")
print("="*60)

# Verificar CSVs
print("\n📄 Archivos CSV:")
csvs_esperados = [
    'metricas_meditadores.csv',
    'comparaciones_completas.csv',
    'resumen_por_meditador.csv',
    'mann_whitney_2grupos.csv',
    'kruskal_wallis_resultados.csv',
    'dunn_posthoc_bonferroni.csv'
]

for csv in csvs_esperados:
    ruta = Path(RESULTS_PATH) / csv
    if ruta.exists():
        tamaño = ruta.stat().st_size / 1024  # KB
        print(f"   ✅ {csv} ({tamaño:.1f} KB)")
    else:
        print(f"   ❌ {csv} - NO ENCONTRADO")

# Verificar PNGs
print("\n📊 Figuras PNG:")
pngs_esperados = [
    '01_senales_ecg_originales.png',
    '02_redes_taichi.png',
    '03_redes_yoga.png',
    '04_redes_sleep_ejemplo.png',
    '05_diferencias_por_meditador.png',
    '06_boxplots_metricas.png',
    '07_kruskal_dunn_completo.png',
    '08_heatmap_diferencias.png',
    '09_comparacion_detallada.png',
    '10_distribucion_grado.png'
]

for png in pngs_esperados:
    ruta = Path(RESULTS_PATH) / png
    if ruta.exists():
        print(f"   ✅ {png}")
    else:
        print(f"   ❌ {png} - NO ENCONTRADO")

# Verificar Excel
print("\n📗 Archivo Excel:")
excel_path = Path(RESULTS_PATH) / 'reporte_completo_redes_ecg.xlsx'
if excel_path.exists():
    tamaño = excel_path.stat().st_size / 1024
    print(f"   ✅ reporte_completo_redes_ecg.xlsx ({tamaño:.1f} KB)")
else:
    print(f"   ⚠️ reporte_completo_redes_ecg.xlsx - No generado (instalar openpyxl)")

# Comparaciones individuales
comp_path = Path(RESULTS_PATH) / 'comparaciones_individuales'
if comp_path.exists():
    num_comp = len(list(comp_path.glob('*.png')))
    print(f"\n📂 Comparaciones individuales: {num_comp} archivos")
else:
    print(f"\n📂 Comparaciones individuales: No generadas")

print("\n" + "="*60)
print("✅ VERIFICACIÓN COMPLETADA")
print("="*60)